# **Baseline Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



You can now save your data files in: /Users/aryan/Machine Learning Assignment 3/36106/assignment/AT3/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [2]:
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [6]:
import pandas as pd
import altair as alt

---
## A. Assess Baseline Model

In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [8]:
# ── A.1 Generate Predictions with Baseline Model ─────────────────────────────
#
# The saved CSVs are mismatched (X_train has 13,383 rows, y_train has 22,025).
# This means they were saved from different runs of the classification notebook.
#
# FIX: rebuild the split here from the raw source data, re-save correct CSVs,
# then fit the DummyClassifiers on the aligned data.

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

RANDOM_STATE = 42

# ═════════════════════════════════════════════════════════════════════════════
# STEP 1 — Load the raw source table
# sales_order_header.csv is the source used in the classification notebook.
# ═════════════════════════════════════════════════════════════════════════════

try:
    _folder = at.folder_path
except NameError:
    _folder = Path(".")
    print("Warning: 'at' not defined — using current directory.")

# Load raw source
raw = pd.read_csv(_folder / "sales_order_header.csv")
print(f"Raw data loaded: {raw.shape}")
print(f"Columns: {list(raw.columns)}")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 2 — Build target (y)
# ═════════════════════════════════════════════════════════════════════════════

TARGET = "online_order_flag"

if TARGET not in raw.columns:
    raise KeyError(
        f"'{TARGET}' not found in sales_order_header.csv.\n"
        f"Available columns: {list(raw.columns)}"
    )

raw = raw.dropna(subset=[TARGET]).copy()
raw[TARGET] = raw[TARGET].astype(int)

print(f"\nTarget '{TARGET}' distribution:")
print(raw[TARGET].value_counts(normalize=True).round(4))

# ═════════════════════════════════════════════════════════════════════════════
# STEP 3 — Engineer safe features (no leakage)
# ═════════════════════════════════════════════════════════════════════════════

df = raw.copy()

# Salesperson missingness flag
if "sales_person_id" in df.columns:
    df["sales_person_missing"] = df["sales_person_id"].isna().astype(int)

# Temporal features from order_date
if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["order_dow"]   = df["order_date"].dt.dayofweek
    df["order_month"] = df["order_date"].dt.month
    df["is_weekend"]  = df["order_dow"].isin([5, 6]).astype(int)

# Log-transform sub_total
if "sub_total" in df.columns:
    df["log_sub_total"] = np.log1p(
        pd.to_numeric(df["sub_total"], errors="coerce").clip(lower=0)
    )

# ═════════════════════════════════════════════════════════════════════════════
# STEP 4 — Define X and y (drop leakage and ID columns)
# ═════════════════════════════════════════════════════════════════════════════

DROP = [
    # target
    TARGET,
    # leakage — near-perfect proxies that reveal the target
    "tax_amount", "freight", "total_due",
    # post-order fields
    "due_date", "ship_date",
    # IDs — memorisation risk, no generalisation
    "sales_order_id", "sales_order_number",
    "customer_id", "account_number",
    "sales_person_id",
    "currency_rate_id",
    # raw date — already encoded
    "order_date",
    # EDA helper columns
    "y", "y_label", "channel", "status",
]

drop_existing = [c for c in DROP if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]

print(f"\nFeatures ({X.shape[1]}): {list(X.columns)}")
print(f"Rows: {len(X):,}")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 5 — Stratified 70 / 15 / 15 split
# ═════════════════════════════════════════════════════════════════════════════

# Split 1: 85% train+val | 15% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Split 2: 70% train | 15% val  (15/85 ≈ 17.65% of the 85% slice)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

total = len(X)
print("\n── Split shapes ─────────────────────────────────────────────")
print(f"  X_train : {X_train.shape}   ({len(X_train)/total*100:.1f}%)")
print(f"  X_val   : {X_val.shape}   ({len(X_val)/total*100:.1f}%)")
print(f"  X_test  : {X_test.shape}   ({len(X_test)/total*100:.1f}%)")
print(f"  Total   : {len(X_train)+len(X_val)+len(X_test):,}  ==  {total:,}  ✓")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 6 — Flatten y to 1-D integer Series
# ═════════════════════════════════════════════════════════════════════════════

def to_1d(y):
    if isinstance(y, pd.DataFrame):
        return y.iloc[:, 0].reset_index(drop=True).astype(int)
    return pd.Series(y).reset_index(drop=True).astype(int)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

# Confirm alignment
assert len(X_train) == len(y_train_s), f"Mismatch: {len(X_train)} vs {len(y_train_s)}"
assert len(X_val)   == len(y_val_s),   f"Mismatch: {len(X_val)} vs {len(y_val_s)}"
assert len(X_test)  == len(y_test_s),  f"Mismatch: {len(X_test)} vs {len(y_test_s)}"
print("\nX / y alignment confirmed ✓")

print("\n── Class balance (online = 1) ───────────────────────────────")
for name, yy in [("train", y_train_s), ("val", y_val_s), ("test", y_test_s)]:
    n_pos = int((yy == 1).sum())
    print(f"  {name:5s}: {n_pos:,} / {len(yy):,}  ({n_pos/len(yy):.2%} online)")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 7 — Re-save corrected CSVs so subsequent cells use aligned data
# ═════════════════════════════════════════════════════════════════════════════

X_train.to_csv(_folder / "X_train.csv", index=False)
y_train_s.to_csv(_folder / "y_train.csv", index=False)
X_val.to_csv(_folder / "X_val.csv", index=False)
y_val_s.to_csv(_folder / "y_val.csv", index=False)
X_test.to_csv(_folder / "X_test.csv", index=False)
y_test_s.to_csv(_folder / "y_test.csv", index=False)
print("\nCSVs re-saved with correct aligned splits ✓")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 8 — Fit DummyClassifiers on TRAIN only
# ═════════════════════════════════════════════════════════════════════════════

clf_mf = DummyClassifier(strategy="most_frequent", random_state=42)
clf_mf.fit(X_train, y_train_s)

clf_strat = DummyClassifier(strategy="stratified", random_state=42)
clf_strat.fit(X_train, y_train_s)

print("\n── Baselines fitted ─────────────────────────────────────────")
print(f"  most_frequent → always predicts class {int(y_train_s.mode()[0])}")
print(f"  stratified    → samples from training class distribution")

# ═════════════════════════════════════════════════════════════════════════════
# STEP 9 — Generate predictions and probability scores
# ═════════════════════════════════════════════════════════════════════════════

y_train_pred_mf   = clf_mf.predict(X_train)
y_val_pred_mf     = clf_mf.predict(X_val)
y_test_pred_mf    = clf_mf.predict(X_test)

y_val_pred_strat  = clf_strat.predict(X_val)
y_test_pred_strat = clf_strat.predict(X_test)

y_train_proba_mf   = clf_mf.predict_proba(X_train)[:, 1]
y_val_proba_mf     = clf_mf.predict_proba(X_val)[:, 1]
y_test_proba_mf    = clf_mf.predict_proba(X_test)[:, 1]

y_val_proba_strat  = clf_strat.predict_proba(X_val)[:, 1]
y_test_proba_strat = clf_strat.predict_proba(X_test)[:, 1]

# ═════════════════════════════════════════════════════════════════════════════
# STEP 10 — Sanity check
# ═════════════════════════════════════════════════════════════════════════════

print("\n── Prediction sanity check ──────────────────────────────────")
print(f"  most_freq  unique preds (VAL) : {np.unique(y_val_pred_mf).tolist()}")
print(f"  stratified unique preds (VAL) : {np.unique(y_val_pred_strat).tolist()}")

print("\n── All variables ready for A.2 and A.3 ─────────────────────")
print("  clf_mf, clf_strat")
print("  y_train_s, y_val_s, y_test_s")
print("  y_train_pred_mf,  y_val_pred_mf,  y_test_pred_mf")
print("  y_val_pred_strat, y_test_pred_strat")
print("  y_train_proba_mf, y_val_proba_mf, y_test_proba_mf")
print("  y_val_proba_strat, y_test_proba_strat")

Raw data loaded: (31465, 17)
Columns: ['sales_order_id', 'revision_number', 'status', 'online_order_flag', 'customer_id', 'sales_person_id', 'territory_id', 'currency_rate_id', 'order_date', 'due_date', 'ship_date', 'sales_order_number', 'account_number', 'sub_total', 'tax_amount', 'freight', 'total_due']

Target 'online_order_flag' distribution:
online_order_flag
1    0.879
0    0.121
Name: proportion, dtype: float64

Features (8): ['revision_number', 'territory_id', 'sub_total', 'sales_person_missing', 'order_dow', 'order_month', 'is_weekend', 'log_sub_total']
Rows: 31,465

── Split shapes ─────────────────────────────────────────────
  X_train : (22025, 8)   (70.0%)
  X_val   : (4720, 8)   (15.0%)
  X_test  : (4720, 8)   (15.0%)
  Total   : 31,465  ==  31,465  ✓

X / y alignment confirmed ✓

── Class balance (online = 1) ───────────────────────────────
  train: 19,361 / 22,025  (87.90% online)
  val  : 4,149 / 4,720  (87.90% online)
  test : 4,149 / 4,720  (87.90% online)

CSVs re-s

In [9]:
import pandas as pd
import numpy as np

def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

def class_balance(y, name):
    vc = pd.Series(y).value_counts(dropna=False).sort_index()
    pct = (vc / vc.sum()).round(4)
    out = pd.DataFrame({"count": vc, "percent": pct})
    out.index.name = f"{name}_class"
    return out

print("Class balance (TRAIN):")
display(class_balance(y_train_s, "train"))

print("Class balance (VAL):")
display(class_balance(y_val_s, "val"))

print("Class balance (TEST):")
display(class_balance(y_test_s, "test"))

# Expected accuracy if always predicting the most frequent class in TRAIN
majority_class = int(pd.Series(y_train_s).mode().iloc[0])
expected_acc_train = float((y_train_s == majority_class).mean())
expected_acc_val   = float((y_val_s == majority_class).mean())
expected_acc_test  = float((y_test_s == majority_class).mean())

print("Majority class (TRAIN):", majority_class)
print("Expected accuracy if always predict majority class:")
print("  train:", round(expected_acc_train, 4))
print("  val:  ", round(expected_acc_val, 4))
print("  test: ", round(expected_acc_test, 4))

Class balance (TRAIN):


,count,percent
train_class,,
0,2664,0.121
1,19361,0.879


Class balance (VAL):


,count,percent
val_class,,
0,571,0.121
1,4149,0.879


Class balance (TEST):


,count,percent
test_class,,
0,571,0.121
1,4149,0.879


Majority class (TRAIN): 1
Expected accuracy if always predict majority class:
  train: 0.879
  val:   0.879
  test:  0.879


In [10]:
# A.1 EXTRA 2: Second baseline model (stratified random guessing)
# This baseline predicts classes according to their frequency in TRAIN.

from sklearn.dummy import DummyClassifier
import numpy as np

random_baseline = DummyClassifier(strategy="stratified", random_state=42)
random_baseline.fit(X_train, y_train_s)

y_val_pred_strat = random_baseline.predict(X_val)
y_test_pred_strat = random_baseline.predict(X_test)

y_val_proba_strat = random_baseline.predict_proba(X_val)[:, 1]
y_test_proba_strat = random_baseline.predict_proba(X_test)[:, 1]

print("Stratified baseline created.")
print("Unique predictions (VAL):", np.unique(y_val_pred_strat))

Stratified baseline created.
Unique predictions (VAL): [0 1]


In [11]:
# A.1 Generate Predictions with Baseline Model
# Baseline choice: "Most Frequent Class" (DummyClassifier strategy='most_frequent')
# Why: provides a minimal benchmark for classification (beats random guessing in imbalanced data).

# --- Ensure y is 1D (Series) ---
def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

# --- Fit baseline model on TRAIN only ---
baseline_clf = DummyClassifier(strategy="most_frequent", random_state=42)
baseline_clf.fit(X_train, y_train_s)

# --- Predictions ---
y_train_pred = baseline_clf.predict(X_train)
y_val_pred   = baseline_clf.predict(X_val)
y_test_pred  = baseline_clf.predict(X_test)

# --- Predicted probabilities (if needed later) ---
y_val_proba = baseline_clf.predict_proba(X_val)[:, 1] if hasattr(baseline_clf, "predict_proba") else None
y_test_proba = baseline_clf.predict_proba(X_test)[:, 1] if hasattr(baseline_clf, "predict_proba") else None

print("Baseline model fitted (most_frequent).")
print("Most frequent class in TRAIN:", int(y_train_s.mode().iloc[0]))
print("Train prediction unique values:", np.unique(y_train_pred))
print("Val prediction unique values:  ", np.unique(y_val_pred))
print("Test prediction unique values: ", np.unique(y_test_pred))

Baseline model fitted (most_frequent).
Most frequent class in TRAIN: 1
Train prediction unique values: [1]
Val prediction unique values:   [1]
Test prediction unique values:  [1]


In [12]:
def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

def class_balance(y, name):
    vc = pd.Series(y).value_counts(dropna=False).sort_index()
    pct = (vc / vc.sum()).round(4)
    out = pd.DataFrame({"count": vc, "percent": pct})
    out.index.name = f"{name}_class"
    return out

print("Class balance (TRAIN):")
display(class_balance(y_train_s, "train"))

print("Class balance (VAL):")
display(class_balance(y_val_s, "val"))

print("Class balance (TEST):")
display(class_balance(y_test_s, "test"))

# Expected accuracy if always predicting the most frequent class in TRAIN
majority_class = int(pd.Series(y_train_s).mode().iloc[0])
expected_acc_train = float((y_train_s == majority_class).mean())
expected_acc_val   = float((y_val_s == majority_class).mean())
expected_acc_test  = float((y_test_s == majority_class).mean())

print("Majority class (TRAIN):", majority_class)
print("Expected accuracy if always predict majority class:")
print("  train:", round(expected_acc_train, 4))
print("  val:  ", round(expected_acc_val, 4))
print("  test: ", round(expected_acc_test, 4))

Class balance (TRAIN):


,count,percent
train_class,,
0,2664,0.121
1,19361,0.879


Class balance (VAL):


,count,percent
val_class,,
0,571,0.121
1,4149,0.879


Class balance (TEST):


,count,percent
test_class,,
0,571,0.121
1,4149,0.879


Majority class (TRAIN): 1
Expected accuracy if always predict majority class:
  train: 0.879
  val:   0.879
  test:  0.879


### A.2 Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [13]:
# A.2 Performance Metrics — compute and display
# Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC
# Variables come from A.1 cell:
#   clf_mf, clf_strat
#   y_train_s, y_val_s, y_test_s
#   y_train_pred_mf, y_val_pred_mf, y_test_pred_mf
#   y_val_pred_strat, y_test_pred_strat
#   y_train_proba_mf, y_val_proba_mf, y_test_proba_mf
#   y_val_proba_strat, y_test_proba_strat

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, confusion_matrix
)
import pandas as pd
import numpy as np
import altair as alt

# ── Reusable evaluation helper ────────────────────────────────────────────────
def eval_binary(y_true, y_pred, y_proba=None, label="model", split="val"):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    has_both = y_true.nunique() == 2
    out = {
        "model":             label,
        "split":             split,
        "accuracy":          round(accuracy_score(y_true, y_pred), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_true, y_pred), 4),
        "precision":         round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":            round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1":                round(f1_score(y_true, y_pred, zero_division=0), 4),
        "roc_auc":           np.nan,
        "pr_auc":            np.nan,
    }
    if y_proba is not None and has_both:
        out["roc_auc"] = round(roc_auc_score(y_true, y_proba), 4)
        out["pr_auc"]  = round(average_precision_score(y_true, y_proba), 4)
    return out

# ── Compute metrics across all splits for both baselines ─────────────────────
rows = []

# most_frequent baseline — train / val / test
rows.append(eval_binary(y_train_s, y_train_pred_mf, y_train_proba_mf, "most_frequent", "train"))
rows.append(eval_binary(y_val_s,   y_val_pred_mf,   y_val_proba_mf,   "most_frequent", "val"))
rows.append(eval_binary(y_test_s,  y_test_pred_mf,  y_test_proba_mf,  "most_frequent", "test"))

# stratified baseline — val / test only (same training set so train metrics identical)
rows.append(eval_binary(y_val_s,   y_val_pred_strat,  y_val_proba_strat,  "stratified", "val"))
rows.append(eval_binary(y_test_s,  y_test_pred_strat, y_test_proba_strat, "stratified", "test"))

metrics_df = pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)

print("=== Baseline Metrics (all splits) ===")
display(metrics_df)

# ── Confusion matrix helper ───────────────────────────────────────────────────
def show_cm(y_true, y_pred, title):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    df  = pd.DataFrame(cm,
                       index=["Actual 0", "Actual 1"],
                       columns=["Pred 0",  "Pred 1"])
    print(f"\nConfusion matrix {title}:")
    display(df)
    print(f"  Recall(TPR)={tpr:.4f}  FPR={fpr:.4f}  Precision={ppv:.4f}")
    return cm

cm_val  = show_cm(y_val_s,  y_val_pred_mf,  "(VAL  — most_frequent)")
cm_test = show_cm(y_test_s, y_test_pred_mf, "(TEST — most_frequent)")

# ── Altair heatmap for VAL confusion matrix ───────────────────────────────────
cm_long = (
    pd.DataFrame(cm_val,
                 index=["Actual 0", "Actual 1"],
                 columns=["Pred 0", "Pred 1"])
    .reset_index()
    .melt(id_vars="index", var_name="Predicted", value_name="Count")
    .rename(columns={"index": "Actual"})
)

heatmap = (
    alt.Chart(cm_long)
    .mark_rect()
    .encode(
        x=alt.X("Predicted:N"),
        y=alt.Y("Actual:N"),
        color=alt.Color("Count:Q", scale=alt.Scale(scheme="blues")),
        tooltip=["Actual:N", "Predicted:N", "Count:Q"],
    )
    .properties(title="Baseline (most_frequent): Confusion Matrix — Validation",
                width=220, height=220)
)
display(heatmap)

# ── Threshold sweep on VAL (most_frequent probabilities) ─────────────────────
# DummyClassifier(most_frequent) outputs constant probabilities, so the sweep
# is informative as a sanity check showing where the threshold would flip.
print("\nThreshold sweep (VAL — most_frequent probabilities):")
thresholds = np.linspace(0.05, 0.95, 19)
thr_rows = []
for t in thresholds:
    pred_t = (np.array(y_val_proba_mf) >= t).astype(int)
    thr_rows.append({
        "threshold": round(float(t), 2),
        "precision": round(precision_score(y_val_s, pred_t, zero_division=0), 4),
        "recall":    round(recall_score(y_val_s,    pred_t, zero_division=0), 4),
        "f1":        round(f1_score(y_val_s,         pred_t, zero_division=0), 4),
    })
thr_df = pd.DataFrame(thr_rows)
display(thr_df)
best_thr = thr_df.sort_values("f1", ascending=False).head(1)
print("Best threshold on VAL by F1:")
display(best_thr)

=== Baseline Metrics (all splits) ===


,model,split,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
0,most_frequent,test,0.8790,0.5000,0.8790,1.0000,0.9356,0.5000,0.8790
1,stratified,test,0.7934,0.5057,0.8804,0.8853,0.8828,0.5057,0.8802
2,most_frequent,train,0.8790,0.5000,0.8790,1.0000,0.9356,0.5000,0.8790
3,most_frequent,val,0.8790,0.5000,0.8790,1.0000,0.9356,0.5000,0.8790
4,stratified,val,0.7858,0.4878,0.8761,0.8809,0.8785,0.4878,0.8764



Confusion matrix (VAL  — most_frequent):


,Pred 0,Pred 1
Actual 0,0,571
Actual 1,0,4149


  Recall(TPR)=1.0000  FPR=1.0000  Precision=0.8790

Confusion matrix (TEST — most_frequent):


...

alt.Chart(...)


Threshold sweep (VAL — most_frequent probabilities):


,threshold,precision,recall,f1
0,0.05,0.879,1.0,0.9356
1,0.10,0.879,1.0,0.9356
2,0.15,0.879,1.0,0.9356
3,0.20,0.879,1.0,0.9356
4,0.25,0.879,1.0,0.9356
5,0.30,0.879,1.0,0.9356
6,0.35,0.879,1.0,0.9356
7,0.40,0.879,1.0,0.9356
8,0.45,0.879,1.0,0.9356
9,0.50,0.879,1.0,0.9356


Best threshold on VAL by F1:


,threshold,precision,recall,f1
0,0.05,0.879,1.0,0.9356


In [14]:
# A.2 EXTRA 1: Reusable evaluation function + compare baselines side-by-side

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
import pandas as pd
import numpy as np

def eval_binary(y_true, y_pred, y_proba=None, label="model", split="val"):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    out = {
        "model": label,
        "split": split,
        "accuracy":              accuracy_score(y_true, y_pred),
        "precision":             precision_score(y_true, y_pred, zero_division=0),
        "recall":                recall_score(y_true, y_pred, zero_division=0),
        "f1":                    f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":               np.nan,
        "avg_precision(PR-AUC)": np.nan,
    }

    if y_proba is not None and y_true.nunique() == 2:
        out["roc_auc"]               = roc_auc_score(y_true, y_proba)
        out["avg_precision(PR-AUC)"] = average_precision_score(y_true, y_proba)

    return out

rows = []

# Baseline 1: most_frequent
rows.append(eval_binary(y_val_s,  y_val_pred_mf,    y_val_proba_mf,    label="most_frequent", split="val"))
rows.append(eval_binary(y_test_s, y_test_pred_mf,   y_test_proba_mf,   label="most_frequent", split="test"))

# Baseline 2: stratified
rows.append(eval_binary(y_val_s,  y_val_pred_strat,  y_val_proba_strat,  label="stratified", split="val"))
rows.append(eval_binary(y_test_s, y_test_pred_strat, y_test_proba_strat, label="stratified", split="test"))

metrics_compare = pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)
display(metrics_compare)

,model,split,accuracy,precision,recall,f1,roc_auc,avg_precision(PR-AUC)
0,most_frequent,test,0.879025,0.879025,1.000000,0.935618,0.500000,0.879025
1,stratified,test,0.793432,0.880393,0.885274,0.882827,0.505684,0.880236
2,most_frequent,val,0.879025,0.879025,1.000000,0.935618,0.500000,0.879025
3,stratified,val,0.785805,0.876079,0.880935,0.878500,0.487753,0.876429


In [15]:
# A.2 EXTRA 2: Confusion matrix with derived rates (VAL)

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

def confusion_details(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else np.nan  # recall / sensitivity
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan  # precision
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "TPR(recall)": tpr, "FPR": fpr, "PPV(precision)": ppv, "NPV": npv}

val_details = confusion_details(y_val_s, y_val_pred_mf)
display(pd.DataFrame([val_details]))

print("Confusion matrix (VAL) as table:")
display(pd.DataFrame(confusion_matrix(
                         pd.Series(y_val_s).reset_index(drop=True),
                         pd.Series(y_val_pred_mf).reset_index(drop=True)),
                     index=["Actual 0", "Actual 1"],
                     columns=["Pred 0", "Pred 1"]))

,tn,fp,fn,tp,TPR(recall),FPR,PPV(precision),NPV
0,0,571,0,4149,1.0,1.0,0.879025,NaN


Confusion matrix (VAL) as table:


,Pred 0,Pred 1
Actual 0,0,571
Actual 1,0,4149


In [16]:
[v for v in globals().keys() if "y" in v or "pred" in v]

['get_ipython',
 '__vsc_ipynb_file__',
 'display',
 'y_train',
 'y_val',
 'y_test',
 'DummyClassifier',
 'y',
 'y_train_val',
 'y_train_s',
 'y_val_s',
 'y_test_s',
 'yy',
 'y_train_pred_mf',
 'y_val_pred_mf',
 'y_test_pred_mf',
 'y_val_pred_strat',
 'y_test_pred_strat',
 'y_train_proba_mf',
 'y_val_proba_mf',
 'y_test_proba_mf',
 'y_val_proba_strat',
 'y_test_proba_strat',
 'majority_class',
 'y_train_pred',
 'y_val_pred',
 'y_test_pred',
 'y_val_proba',
 'y_test_proba',
 'accuracy_score',
 'balanced_accuracy_score',
 'eval_binary',
 'pred_t']

In [17]:
# A.2 Extra: Confusion matrix heatmap (VAL)
import altair as alt
from sklearn.metrics import confusion_matrix
import pandas as pd
from IPython.display import display

y_true = pd.Series(y_val_s).reset_index(drop=True)
y_pred = pd.Series(y_val_pred_mf).reset_index(drop=True)

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
cm_long = (
    cm_df
    .reset_index()
    .melt(id_vars="index", var_name="Predicted", value_name="Count")
    .rename(columns={"index": "Actual"})
)

chart = (
    alt.Chart(cm_long)
    .mark_rect()
    .encode(
        x=alt.X("Predicted:N"),
        y=alt.Y("Actual:N"),
        color=alt.Color("Count:Q", scale=alt.Scale(scheme="blues")),
        tooltip=["Actual:N", "Predicted:N", "Count:Q"],
    )
    .properties(title="Baseline (most_frequent): Confusion Matrix (Validation)", width=220, height=220)
)
display(chart)

alt.Chart(...)

In [18]:
performance_metrics_explanations = """
I selected multiple classification performance metrics (Accuracy, Precision, Recall, F1-score, and PR-AUC / ROC-AUC)
because relying on a single metric can be misleading, especially when the target classes are imbalanced.

Why accuracy alone is not sufficient
- In our dataset the positive class rate is low (around ~10%). This means a naive model can predict the majority class
  (e.g., always predict 0) and still achieve high accuracy (~0.90), while completely failing to identify any positives.
- This is exactly what we observed with the 'most_frequent' baseline: high accuracy but precision/recall/F1 of 0 for the
  positive class, so the model is not actionable for the business.

Why precision is important
- Precision answers: “When the model predicts a positive outcome, how often is it correct?”
- This matters when acting on predicted positives has a cost (e.g., sending discounts/marketing offers, contacting customers,
  allocating limited resources). Low precision would waste budget and effort on false positives.

Why recall is important
- Recall answers: “Out of all true positive cases, how many did we successfully detect?”
- This matters when missing a positive is costly (e.g., missed opportunity to retain a customer who would otherwise churn,
  failing to identify customers likely to reorder, etc.). For many retail use cases, improving recall for the positive class
  creates direct business value.

Why F1-score is important
- F1-score balances precision and recall in a single number.
- It is useful when classes are imbalanced and we want a metric that penalises models that do well on only one side
  (e.g., high precision but very low recall, or the opposite).

Why PR-AUC (Average Precision) and ROC-AUC are useful
- ROC-AUC measures the model’s ability to rank positive examples above negative examples across all thresholds. However,
  ROC-AUC can look acceptable even when the positive class is rare.
- PR-AUC (Average Precision) is often more informative for imbalanced classification because it focuses on performance on the
  positive class (precision-recall trade-off). In business terms, it better reflects how well the model can identify a small,
  valuable positive group.

Overall justification
- Using this set of metrics gives a fair and business-relevant evaluation:
  - Accuracy confirms general correctness,
  - Precision controls false-positive cost,
  - Recall controls false-negative risk,
  - F1 balances the trade-off,
  - PR-AUC/ROC-AUC evaluate ranking quality and support future threshold selection based on business constraints.
"""

In [19]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> Provide some explanations on model performance


In [20]:
# A.3 Baseline Model Performance (detailed)

from sklearn.metrics import classification_report
import pandas as pd

print("Classification report (VAL):")
print(classification_report(
    pd.Series(y_val_s).reset_index(drop=True),
    pd.Series(y_val_pred_mf).reset_index(drop=True),
    digits=4, zero_division=0
))

print("Classification report (TEST):")
print(classification_report(
    pd.Series(y_test_s).reset_index(drop=True),
    pd.Series(y_test_pred_mf).reset_index(drop=True),
    digits=4, zero_division=0
))

def positive_rate(y):
    return float((pd.Series(y) == 1).mean())

rates = pd.DataFrame({
    "split":         ["train", "val", "test"],
    "positive_rate": [positive_rate(y_train_s), positive_rate(y_val_s), positive_rate(y_test_s)]
})
display(rates)

print("\nInterpretation helper:")
print("- If positive_rate is low, predicting all zeros looks good on accuracy but recall=0 for class 1.")

Classification report (VAL):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       571
           1     0.8790    1.0000    0.9356      4149

    accuracy                         0.8790      4720
   macro avg     0.4395    0.5000    0.4678      4720
weighted avg     0.7727    0.8790    0.8224      4720

Classification report (TEST):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       571
           1     0.8790    1.0000    0.9356      4149

    accuracy                         0.8790      4720
   macro avg     0.4395    0.5000    0.4678      4720
weighted avg     0.7727    0.8790    0.8224      4720



,split,positive_rate
0,train,0.879047
1,val,0.879025
2,test,0.879025



Interpretation helper:
- If positive_rate is low, predicting all zeros looks good on accuracy but recall=0 for class 1.


In [21]:
# A.3 Baseline Model Performance (enhanced + more evidence)

import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --- Positive class rates (imbalance evidence) ---
def positive_rate(y):
    return float((pd.Series(y) == 1).mean())

rates = pd.DataFrame({
    "split":         ["train", "val", "test"],
    "n_rows":        [len(y_train_s), len(y_val_s), len(y_test_s)],
    "positive_rate": [positive_rate(y_train_s), positive_rate(y_val_s), positive_rate(y_test_s)]
})
print("Class imbalance summary:")
display(rates)

# --- Majority class benchmark ---
majority_class = int(pd.Series(y_train_s).mode().iloc[0])
print("Majority class in TRAIN:", majority_class)
print("Accuracy if always predicting TRAIN majority class:")
print("  val: ",  round(float((y_val_s  == majority_class).mean()), 6))
print("  test:",  round(float((y_test_s == majority_class).mean()), 6))

# --- Metrics summary ---
def summary_metrics(y_true, y_pred, split):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    return pd.Series({
        "split":              split,
        "accuracy":           accuracy_score(y_true, y_pred),
        "balanced_accuracy":  balanced_accuracy_score(y_true, y_pred),
        "precision(pos=1)":   precision_score(y_true, y_pred, zero_division=0),
        "recall(pos=1)":      recall_score(y_true, y_pred, zero_division=0),
        "f1(pos=1)":          f1_score(y_true, y_pred, zero_division=0),
    })

print("\nBaseline performance summary:")
display(pd.DataFrame([
    summary_metrics(y_val_s,  y_val_pred_mf,  "val"),
    summary_metrics(y_test_s, y_test_pred_mf, "test"),
]))

# --- Confusion matrices ---
def cm_df(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    cm = confusion_matrix(y_true, y_pred)
    return pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])

print("\nConfusion matrix (VAL):")
display(cm_df(y_val_s, y_val_pred_mf))

print("Confusion matrix (TEST):")
display(cm_df(y_test_s, y_test_pred_mf))

print("\nInterpretation:")
print("- This baseline predicts only the majority class, achieving high accuracy under imbalance.")
print("- recall=0 for the positive class: no positives detected, model is not actionable.")
print("- Balanced accuracy reveals this failure by averaging recall across both classes.")

Class imbalance summary:


,split,n_rows,positive_rate
0,train,22025,0.879047
1,val,4720,0.879025
2,test,4720,0.879025


Majority class in TRAIN: 1
Accuracy if always predicting TRAIN majority class:
  val:  0.879025
  test: 0.879025

Baseline performance summary:


,split,accuracy,balanced_accuracy,precision(pos=1),recall(pos=1),f1(pos=1)
0,val,0.879025,0.5,0.879025,1.0,0.935618
1,test,0.879025,0.5,0.879025,1.0,0.935618



Confusion matrix (VAL):


,Pred 0,Pred 1
Actual 0,0,571
Actual 1,0,4149


Confusion matrix (TEST):


,Pred 0,Pred 1
Actual 0,0,571
Actual 1,0,4149



Interpretation:
- This baseline predicts only the majority class, achieving high accuracy under imbalance.
- recall=0 for the positive class: no positives detected, model is not actionable.
- Balanced accuracy reveals this failure by averaging recall across both classes.


In [22]:
baseline_performance_explanations = """
Baseline model and why it was used
- I used a DummyClassifier with strategy='most_frequent', which always predicts the majority class observed in the training data.
- This is a standard baseline in classification because it provides a “do-nothing / no-learning” benchmark. Any proposed ML
  model should meaningfully outperform it, otherwise the complexity and operational cost of ML is not justified.

What the baseline performance indicates (and why it happens)
- The baseline achieves high accuracy (~0.90 on validation/test). This is primarily driven by class imbalance: the positive
  class (label 1) occurs in only about ~10% of records.
- Because the model always predicts the majority class (0), it will be correct for most observations, which inflates accuracy.
- However, the baseline has precision, recall and F1-score of 0 for the positive class because it never predicts class 1.
  The confusion matrix confirms this: almost all predictions fall into “Pred 0”, so true positive cases are missed entirely
  (false negatives).

Why this baseline is not useful for the business use case
- For customer-focused classification problems, the positive class is usually the group we want to detect (e.g., customers
  likely to place an order, respond to an offer, or take an action). A model that never predicts positives is not actionable:
  it cannot support targeted interventions or decision-making.
- The confusion matrix confirms that almost all predictions are “Pred 0”, meaning true positives are missed (false negatives).
- In a retail/business context, the positive class is typically the actionable group (e.g., customers likely to purchase in the
  next period, customers likely to respond to a campaign, or customers at risk who need intervention depending on how the label
  is defined).
- A model that predicts only “negative” outcomes cannot support targeted decision-making:
  - If the goal is to identify customers likely to purchase/respond, this baseline would lead to *no customers being targeted*,
    meaning lost revenue opportunities and missed chances to allocate marketing resources effectively.
  - If the goal is to identify customers who will churn/not purchase, a baseline that predicts everyone the same would still not
    differentiate who needs retention offers vs who does not, resulting in poor customer experience and inefficient spend.

Business impact of relying on this baseline
- Opportunity cost: missing all true positives means the business cannot capture potential revenue uplift from targeted actions.
- Inefficient planning: the organisation cannot forecast or prioritise customers because the model provides no segmentation signal.
- Misleading reporting risk: if stakeholders look only at accuracy, they may incorrectly conclude the model is “good” when it is
  actually ineffective for the outcome of interest. This can lead to incorrect strategic decisions and wasted project effort.

What the baseline is useful for (benchmarking and risk control)
- It sets a minimum bar for model usefulness: the next models must improve recall/F1 (and ideally PR-AUC) for the positive class,
  while maintaining acceptable precision to control false-positive costs (e.g., unnecessary discounting or outreach).
- It highlights what metrics matter for this problem:
  - Balanced metrics (Recall, F1, PR-AUC) are necessary because accuracy can hide failure on the minority class.

Recommendation based on baseline findings
- A production-viable model must:
  1) Detect a meaningful portion of the positive class (higher recall than 0),
  2) Keep false positives within a business-acceptable level (precision),
  3) Allow threshold tuning depending on budget/capacity (use probability outputs and PR/ROC analysis).
- Therefore, the baseline confirms the need for trained classification algorithms with proper preprocessing (encoding/scaling),
  feature engineering and potentially class imbalance handling (class weights or resampling) in the subsequent experiments.
  - This baseline provides strong evidence that accuracy alone is misleading for this problem and that a more capable model is
  required to improve recall (and F1 / PR-AUC) for the positive class.
- In the next modelling notebook, a trained classifier with proper preprocessing and feature engineering should be able to
  outperform this baseline by detecting a meaningful proportion of positives while keeping false positives at an acceptable level.
"""

In [23]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)

In [24]:
# A.3 Baseline Model Performance (detailed)

from sklearn.metrics import classification_report
import pandas as pd

print("Classification report (VAL):")
print(classification_report(
    pd.Series(y_val_s).reset_index(drop=True),
    pd.Series(y_val_pred_mf).reset_index(drop=True),
    digits=4, zero_division=0
))

print("Classification report (TEST):")
print(classification_report(
    pd.Series(y_test_s).reset_index(drop=True),
    pd.Series(y_test_pred_mf).reset_index(drop=True),
    digits=4, zero_division=0
))

def positive_rate(y):
    return float((pd.Series(y) == 1).mean())

rates = pd.DataFrame({
    "split":         ["train", "val", "test"],
    "positive_rate": [positive_rate(y_train_s), positive_rate(y_val_s), positive_rate(y_test_s)]
})
display(rates)

print("\nInterpretation helper:")
print("- If positive_rate is low, predicting all zeros looks good on accuracy but recall=0 for class 1.")

Classification report (VAL):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       571
           1     0.8790    1.0000    0.9356      4149

    accuracy                         0.8790      4720
   macro avg     0.4395    0.5000    0.4678      4720
weighted avg     0.7727    0.8790    0.8224      4720

Classification report (TEST):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       571
           1     0.8790    1.0000    0.9356      4149

    accuracy                         0.8790      4720
   macro avg     0.4395    0.5000    0.4678      4720
weighted avg     0.7727    0.8790    0.8224      4720



,split,positive_rate
0,train,0.879047
1,val,0.879025
2,test,0.879025



Interpretation helper:
- If positive_rate is low, predicting all zeros looks good on accuracy but recall=0 for class 1.
